# 08 — Single Rigid Body Dynamics

MPCの中心である並進・回転運動方程式を、手計算とCasADi実装で対応づけます。

**前提**: `07_swing_trajectory.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebook_pympc":
    ROOT = ROOT.parent
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
if str(PYMPC_ROOT) not in sys.path:
    sys.path.insert(0, str(PYMPC_ROOT))

os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
os.environ.setdefault("MUJOCO_GL", "egl")
print("workspace :", ROOT)
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 並進

\[
m\dot v=\sum_{i=1}^{4}c_iF_i+F_{ext}+mg
\]

## 回転

\[
I\dot\omega+\omega\times I\omega
  =R_{B\leftarrow W}\left(\sum_i c_i(r_i-p)\times F_i+\tau_{ext}\right)
\]

SRBDは脚の質量分布や関節運動を直接予測せず、それらを胴体の質量・慣性と
接触力へ縮約します。

In [2]:
import numpy as np

# centroidal_model_nominal.py::forward_dynamicsの並進部と同じ変数。
mass, g = 15.019, 9.81                 # m [kg], g [m/s^2]
contacts = np.array([1, 1, 1, 1])      # c_i: 接地=1, 遊脚=0
forces = np.tile([0., 0., mass*g/4], (4, 1))  # F_i [N], 4脚×xyz

# 実装対応: linear_com_acc = (1/mass)*sum(c_i*F_i) + [0,0,-g]
# 数式対応: m*v_dot = sum(c_i F_i) + m*g_vector
net_contact_force = (contacts[:, None] * forces).sum(axis=0)
acc = net_contact_force / mass + np.array([0, 0, -g])
print("balanced acceleration:", acc, "m/s^2")

# FLへ+20 Nの水平力を加え、a_x=20/mになることを検算する。
forces[0, 0] = 20.0
net_contact_force = (contacts[:, None] * forces).sum(axis=0)
acc2 = net_contact_force / mass + np.array([0, 0, -g])
print("after +20N at FL:", np.round(acc2, 3), "m/s^2")
assert np.allclose(acc, 0)
assert np.isclose(acc2[0], 20.0/mass)

balanced acceleration: [0.00000000e+00 0.00000000e+00 1.77635684e-15] m/s^2
after +20N at FL: [1.332 0.    0.   ] m/s^2


In [3]:
from quadruped_pympc.controllers.gradient.nominal.centroidal_model_nominal import Centroidal_Model_Nominal
model = Centroidal_Model_Nominal()
print("CasADi state dim:", model.states.size1())
print("CasADi input dim:", model.inputs.size1())
print("reference dim   :", model.y_ref.size1())
assert model.states.size1() == 30
assert model.inputs.size1() == 24

CasADi state dim: 30
CasADi input dim: 24
reference dim   : 54


実装では接触フラグが力とモーメントの両方をmaskします。
遊脚に大きなGRF変数が入っても運動へ寄与しませんが、後段Interfaceでも再度maskされます。
二重防御の場所を区別してください。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。